### =====================
### Credit Operations ETL
###=====================

New operations along with their credit risk (probability of default)

In [0]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import os
import mlflow
from pyspark.sql import functions as F
import warnings
from credit_risk.feature_engineering import feature_engineering
warnings.filterwarnings('ignore')

# Definition of the final table saved
env = "prd"
catalog_name = f"mlops_{env}"
schema_name = "dbk_credit_analytics"
table_name = "operations_credit_risk"

# Definition of the used model in production
model_catalog_name = "mlops_prd"
model_schema_name = "dbk_credit_risk"
model_name = "credit_risk_model_custom"
pyfunc_model_name = f"{model_catalog_name}.{model_schema_name}.{model_name}"
model_uri = f"models:/{pyfunc_model_name}@latest-model"

# Final features that need to be passed to the model as inputs
final_features = [
    "int_rate",
    "sub_grade",
    "annual_inc",
    "inq_last_6mths",
    "total_rev_hi_lim",
    "purpose",
    "dti",
    "home_ownership",
    "initial_list_status",
    "verification_status",
    "mths_since_earliest_cr_line",
    "addr_state"
]

Importing the data to have its probability of default predicted

In [0]:
# selecting the records IDs and the columns neeed for the model's input
columns_selected_final = ['id', 'member_id'] + final_features

# Extracting data that represents new operations
df_raw = spark.table("bigquery_credit_analytics_catalog.credit_analytics.loan_data")

num_records = df_raw.count()
print(f"Number of records in the dataset: {num_records}")

# Applies the feature engineering necessary for the inputs of the model
df = feature_engineering(df_raw)

# For this part, to represent the newest that, we are selecting only the data from 2015
df["issue_d_date"] = pd.to_datetime(df["issue_d_date"])
df = df[df["issue_d_date"].dt.year == 2015]
df = df[columns_selected_final].reset_index(drop=True)

Adapting the environment to be able to run the model

In [0]:
# In order to use the model, we need to install the necessary libs and modules
# for it to be loaded and used.

# Importing the necessary whl to be installed from the model's artifact
local_path = mlflow.artifacts.download_artifacts(model_uri)
wheel_dir = os.path.join(local_path, "code")

wheel_file = [
    f for f in os.listdir(wheel_dir)
    if f.endswith(".whl")
][0]

# Installing the necessary modules and libs obtained from the model's artifacts
wheel_path = os.path.join(wheel_dir, wheel_file)
%pip install {wheel_path} --quiet

Predicting the probabilities

In [0]:
# Loading the model
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Predicting
predictions = loaded_model.predict(pd.DataFrame(df[final_features].astype(str)))

Preparing the final table with the operations, its features and the probability of default

In [0]:
# Building the dataframe with the probabilities
df_pred = pd.DataFrame(pd.concat([df, pd.DataFrame(predictions)], axis=1))
df_pred_spark = spark.createDataFrame(df_pred).withColumnRenamed("Probability of Default", "probability_default")

# Final operation features to be added to final table
final_columns = ["id", "member_id", "loan_amnt", "funded_amnt", "issue_d", "purpose", "total_pymnt", "int_rate", "addr_state", "home_ownership", "emp_length", "annual_inc", "earliest_cr_line", "term"]

# Joining the predictions with the original dataset
cond_j = [df_pred_spark.id == df_raw.id, df_pred_spark.member_id == df_raw.member_id]
df_final = df_pred_spark.join(df_raw, cond_j, how="inner") \
                        .select(
                            *[df_raw[c] for c in final_columns],
                            df_pred_spark["probability_default"]
                        )

Saving the dataframe to unity catalog

In [0]:
df_final.write.mode("overwrite").saveAsTable(
    f"{catalog_name}.{schema_name}.{table_name}"
)